In [ ]:
# ══════════════════════════════════════════════
# SETUP COLAB: MOUNT DRIVE + GIẢI NÉN DATA
# ══════════════════════════════════════════════
from google.colab import drive
import os, zipfile, glob

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/Data2.zip'

if not os.path.exists('/content/Data2'):
    print('Đang giải nén Data2.zip ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Giải nén xong!')
else:
    print('Data2 đã tồn tại, bỏ qua giải nén.')

npz_files = glob.glob('/content/Data2/**/*.npz', recursive=True)
print(f'Tìm thấy {len(npz_files)} file .npz trong /content/Data2')

if len(npz_files) == 0:
    npz_files2 = glob.glob('/content/Data2/Data2/**/*.npz', recursive=True)
    if npz_files2:
        print(f'  → Phát hiện thư mục lồng, tìm thấy {len(npz_files2)} file tại /content/Data2/Data2')
        print('  → Đã tự động điều chỉnh DATA_PATH bên dưới.')
    else:
        print('  ⚠ Không tìm thấy file .npz! Kiểm tra lại ZIP_PATH hoặc cấu trúc zip.')
        import os
        for root, dirs, files in os.walk('/content/Data2'):
            print(root, '->', files[:3])


In [1]:
import tensorflow as tf
import numpy as np
import os
import glob
import json
import math
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, BatchNormalization,
    Bidirectional, Layer
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')



TensorFlow version: 2.15.0
GPU available: False


In [ ]:
# ══════════════════════════════════════════════
# CẤU HÌNH — chỉnh tại đây (v3: mô hình nhỏ cho 20-30 lớp)
# ══════════════════════════════════════════════

import glob as _glob, os as _os
_candidate1 = '/content/Data2'
_candidate2 = '/content/Data2/Data2'
if _glob.glob(_candidate2 + '/**/*.npz', recursive=True):
    DATA_PATH = _candidate2
else:
    DATA_PATH = _candidate1
print(f'DATA_PATH = {DATA_PATH}')

LABEL_MAP_PATH  = '/content/drive/MyDrive/Logs/label_map.json'
CHECKPOINT_DIR  = '/content/drive/MyDrive/Models/checkpoints'
LOG_DIR         = '/content/drive/MyDrive/Models/logs'
MODEL_SAVE_PATH = '/content/drive/MyDrive/Models/final_model.keras'
PLOT_DIR        = '/content/drive/MyDrive/Models'

SEQUENCE_LEN    = 60
N_FEATURES      = 201
BATCH_SIZE      = 32
EPOCHS          = 60        # [v3] ít lớp hơn -> hội tụ nhanh, cho thêm epoch + EarlyStopping lo phần dừng đúng lúc
VAL_SPLIT       = 0.1
TEST_SPLIT      = 0.1
WARMUP_EPOCHS   = 3          # [v3] model nhỏ, warmup ngắn hơn là đủ
BASE_LR         = 1e-3
MIN_LR          = 1e-6

# ── Tham số mô hình (v3: thu nhỏ cho bài toán 20-30 lớp) ────────────────
# So với v2 (3 tầng BiLSTM 128 units, dense 256/128, L2=5e-4, dropout cao):
# với ít lớp hơn nhiều, không gian quyết định đơn giản hơn hẳn -> mô hình
# to cỡ đó rất dễ overfit nếu số sample/lớp không cực lớn. Giảm quy mô +
# giảm regularization tương ứng (ít tham số hơn thì không cần "phanh" mạnh
# bằng dropout/L2 cao như trước).
LSTM_UNITS_1    = 64          # tầng BiLSTM thứ nhất
LSTM_UNITS_2    = 64          # tầng BiLSTM thứ hai (không còn tầng 3)
DENSE_1_UNITS   = 128         # chỉ còn 1 tầng Dense trong classifier head
L2_REG          = 1e-4        # giảm vì mô hình đã nhỏ hơn nhiều
LSTM_DROPOUT    = 0.2          # giảm nhẹ so với 0.3
RECURRENT_DROP  = 0.1
DENSE_DROPOUT   = 0.3          # giảm nhẹ so với 0.4
LABEL_SMOOTHING = 0.1
AUGMENT_NOISE   = 0.012
AUGMENT_SCALE   = 0.10
AUTOTUNE        = tf.data.AUTOTUNE

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)


In [3]:
# ══════════════════════════════════════════════
# LOAD LABEL MAP & DANH SÁCH FILE
# ══════════════════════════════════════════════
with open(LABEL_MAP_PATH, 'r', encoding='utf-8') as f:
    label_map = json.load(f)

NUM_CLASSES = len(label_map)
print(f'Số nhãn: {NUM_CLASSES}')

all_files = glob.glob(os.path.join(DATA_PATH, '**', '*.npz'), recursive=True)
print(f'Tổng số mẫu: {len(all_files)}')

def get_action_label(path):
    return os.path.basename(os.path.dirname(path))

# [FIX-LEAKAGE] Đọc trường 'video' (tên video gốc) được ghi trong từng
# .npz bởi create_data_augment.py đã patch. Các bản augment sinh ra từ
# CÙNG một video sẽ có cùng group -> đảm bảo được gom trọn vào 1 tập
# (train HOẶC val HOẶC test), không bao giờ bị chia sang 2 tập khác nhau.
# Nếu file cũ (tạo trước khi patch) không có trường 'video', fallback về
# dùng chính tên file làm group riêng (không nhóm được -> vẫn có thể
# leakage cho phần dữ liệu cũ này, cần tạo lại data nếu muốn an toàn 100%).
def get_video_group(path):
    try:
        with np.load(path) as d:
            if 'video' in d.files:
                return str(d['video'])
    except Exception:
        pass
    return os.path.basename(path)

strat_labels = [get_action_label(p) for p in all_files]
groups       = [get_video_group(p) for p in all_files]

n_no_group = sum(1 for p, g in zip(all_files, groups) if g == os.path.basename(p))
if n_no_group:
    print(f"⚠ {n_no_group}/{len(all_files)} file không có trường 'video' "
          f"(dữ liệu cũ, tạo trước khi patch) -> không group được, "
          f"vẫn có nguy cơ leakage cho các file này.")
else:
    print("✅ Tất cả file đều có group video -> split an toàn theo video gốc.")

from sklearn.model_selection import GroupShuffleSplit

# Bước 1: tách TRAIN vs TEMP(val+test) theo GROUP (video gốc), không theo
# từng sample riêng lẻ như trước.
gss1 = GroupShuffleSplit(n_splits=1, test_size=VAL_SPLIT + TEST_SPLIT, random_state=42)
train_idx, temp_idx = next(gss1.split(all_files, groups=groups))

train_files = [all_files[i] for i in train_idx]
temp_files  = [all_files[i] for i in temp_idx]
temp_groups = [groups[i]    for i in temp_idx]

# Bước 2: tách TEMP thành VAL / TEST, vẫn theo GROUP để không rò rỉ.
gss2 = GroupShuffleSplit(n_splits=1, test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT),
                         random_state=42)
val_idx, test_idx = next(gss2.split(temp_files, groups=temp_groups))

val_files  = [temp_files[i] for i in val_idx]
test_files = [temp_files[i] for i in test_idx]

print(f'  Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}')

# Ghi chú: GroupShuffleSplit không hỗ trợ stratify đồng thời với group,
# nên phân bố class giữa các tập có thể lệch nhẹ so với bản gốc (stratify
# thuần theo sample). In ra để kiểm tra thủ công; nếu lệch nhiều, cân nhắc
# StratifiedGroupKFold (sklearn >= 1.1) thay vì GroupShuffleSplit.
import collections
for name, files in [('Train', train_files), ('Val', val_files), ('Test', test_files)]:
    dist = collections.Counter(get_action_label(p) for p in files)
    missing = [a for a in label_map if dist.get(a, 0) == 0]
    if missing:
        print(f"  ⚠ {name}: thiếu hoàn toàn {len(missing)} lớp: {missing[:5]}{'...' if len(missing) > 5 else ''}")

# ── [FIX] Kiểm tra data leakage ở CẢ 2 mức: file VÀ group (video gốc) ──
train_set = set(train_files)
val_set   = set(val_files)
test_set  = set(test_files)
assert len(train_set & val_set)  == 0, "⚠ DATA LEAKAGE (file): train ∩ val"
assert len(train_set & test_set) == 0, "⚠ DATA LEAKAGE (file): train ∩ test"
assert len(val_set   & test_set) == 0, "⚠ DATA LEAKAGE (file): val ∩ test"

path2group    = dict(zip(all_files, groups))
train_groups  = set(path2group[p] for p in train_files)
val_groups_s  = set(path2group[p] for p in val_files)
test_groups_s = set(path2group[p] for p in test_files)
assert len(train_groups & val_groups_s)  == 0, "⚠ DATA LEAKAGE (video gốc): train ∩ val"
assert len(train_groups & test_groups_s) == 0, "⚠ DATA LEAKAGE (video gốc): train ∩ test"
assert len(val_groups_s & test_groups_s) == 0, "⚠ DATA LEAKAGE (video gốc): val ∩ test"
print("✅ Không có data leakage, kể cả ở cấp video gốc, giữa train / val / test.")


Số nhãn: 6
Tổng số mẫu: 5829
  Train: 4663 | Val: 583 | Test: 583


In [4]:
# ══════════════════════════════════════════════
# TF.DATA PIPELINE + DATA AUGMENTATION
# ══════════════════════════════════════════════

def _load_npz(path):
    data = np.load(path.decode('utf-8'))
    seq  = data['sequence'].astype(np.float32)
    lbl  = np.int32(data['label'])
    return seq, lbl

def parse_fn(path):
    seq, lbl = tf.numpy_function(_load_npz, [path], [tf.float32, tf.int32])
    seq.set_shape([SEQUENCE_LEN, N_FEATURES])
    lbl.set_shape([])
    return seq, lbl

def one_hot_fn(seq, lbl):
    """Chuyển integer label → one-hot để dùng CategoricalCrossentropy."""
    return seq, tf.one_hot(lbl, NUM_CLASSES)

def augment_fn(seq, lbl):
    # [FIX-ACCURACY] mask=1 tai nhung diem THUC SU duoc MediaPipe phat hien
    # (khac 0.0), mask=0 tai cac landmark "mat" (sentinel = 0.0). Dung de
    # khong lam hong tin hieu "khong phat hien duoc" khi augment.
    mask = tf.cast(tf.not_equal(seq, 0.0), tf.float32)

    # 1. Gaussian noise - CHI cong vao diem thuc su duoc phat hien.
    #    [FIX] Ban goc cong nhieu vao CA landmark mat (gia tri sentinel
    #    0.0), lam mo tin hieu "khong phat hien duoc" ma model co the dang
    #    dua vao de phan biet cac ky hieu.
    noise = tf.random.normal(tf.shape(seq), mean=0.0, stddev=AUGMENT_NOISE)
    seq = seq + noise * mask

    # 2. Random global scale (+-AUGMENT_SCALE), quanh TAM KHUNG HINH
    #    (0.5, 0.5, 0.0) thay vi quanh GOC TOA DO (0,0,0).
    #    [FIX-CRITICAL] Ban goc lam `seq = seq * scale`: vi toa do da chuan
    #    hoa deu la so DUONG (khong nam quanh 0), nhan truc tiep voi scale
    #    KHONG tuong duong voi zoom quanh tam co the, ma tuong duong voi
    #    dich chuyen ve phia goc tren-trai anh (goc toa do) voi muc do lech
    #    nhau tuy theo khoang cach tu goc toa do -> bien "scale" thanh
    #    nhieu dich chuyen meo, khong giong bat ky hieu ung camera thuc te
    #    nao. Loi nay anh huong MOI batch, MOI epoch trong suot qua trinh
    #    train (khong chi mot phan du lieu augment offline) nen tac dong
    #    toi accuracy la dang ke nhat trong ca pipeline.
    scale = tf.random.uniform([], 1.0 - AUGMENT_SCALE, 1.0 + AUGMENT_SCALE)
    center = tf.tile(tf.constant([0.5, 0.5, 0.0], dtype=tf.float32),
                      [N_FEATURES // 3])
    scaled = (seq - center) * scale + center
    seq = tf.where(mask > 0, scaled, seq)   # giu nguyen landmark "mat"

    # 3. Temporal shift: dich theo truc thoi gian, LAP LAI FRAME BIEN thay
    #    vi cuon vong tron.
    #    [FIX] tf.roll la cuon VONG TRON: vai frame dau chuoi bi "chuyen"
    #    ra cuoi chuoi (va nguoc lai), tao mot buoc nhay tu the phi thuc te
    #    ngay giua chuoi (thay vi chi dich thoi gian don thuan). Nay dung
    #    tf.gather voi index bi clip ve bien de LAP LAI frame dau/cuoi
    #    thay cho phan bi "day ra ngoai", giong pad-edge hon la wrap-around.
    shift = tf.random.uniform([], -SEQUENCE_LEN // 10, SEQUENCE_LEN // 10 + 1,
                              dtype=tf.int32)
    T = tf.shape(seq)[0]
    idx = tf.clip_by_value(tf.range(T) - shift, 0, T - 1)
    seq = tf.gather(seq, idx, axis=0)

    return seq, lbl

def make_dataset(file_list, shuffle=False, repeat=False,
                 cache=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices(file_list)
    if shuffle:
        ds = ds.shuffle(max(1, len(file_list)), reshuffle_each_iteration=True)
    if repeat:
        ds = ds.repeat()
    ds = ds.map(parse_fn,   num_parallel_calls=AUTOTUNE)
    ds = ds.map(one_hot_fn, num_parallel_calls=AUTOTUNE)  # one-hot labels
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=AUTOTUNE)  # chỉ train
    if cache:
        ds = ds.cache()
    # [FIX] Bỏ drop_remainder=repeat → không làm mất batch cuối của train
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_files, shuffle=True,  repeat=True,
                        cache=False, augment=True)   # augment CHỈ cho train
val_ds   = make_dataset(val_files,   shuffle=False, repeat=False,
                        cache=True,  augment=False)
test_ds  = make_dataset(test_files,  shuffle=False, repeat=False,
                        cache=True,  augment=False)

# [FIX] Dùng math.ceil thay vì // để không bỏ sót mẫu cuối
steps_per_epoch  = max(1, math.ceil(len(train_files) / BATCH_SIZE))
validation_steps = max(1, math.ceil(len(val_files)   / BATCH_SIZE))
print(f'steps_per_epoch={steps_per_epoch} | validation_steps={validation_steps}')


steps_per_epoch=145 | validation_steps=18


In [ ]:
# ══════════════════════════════════════════════
# TEMPORAL ATTENTION LAYER (v2: thêm L2 regularizer)
# ══════════════════════════════════════════════
class TemporalAttention(Layer):
    """
    Soft attention theo trục thời gian.
    Input : (batch, T, D)
    Output: (batch, D)
    v2: thêm L2 regularizer trên W_a và v_a để tránh overfit.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        D = input_shape[-1]
        self.W_a = self.add_weight(
            name='W_a', shape=(D, D),
            initializer='glorot_uniform',
            regularizer=l2(L2_REG),       
            trainable=True
        )
        self.v_a = self.add_weight(
            name='v_a', shape=(D, 1),
            initializer='glorot_uniform',
            regularizer=l2(L2_REG),       
            trainable=True
        )
        super().build(input_shape)

    def call(self, hidden_states):
        score  = tf.nn.tanh(tf.matmul(hidden_states, self.W_a))
        score  = tf.matmul(score, self.v_a)
        score  = tf.squeeze(score, axis=-1)
        alpha  = tf.nn.softmax(score, axis=-1)
        alpha  = tf.expand_dims(alpha, axis=-1)
        return tf.reduce_sum(hidden_states * alpha, axis=1)

    def get_config(self):
        return super().get_config()


In [ ]:
# ══════════════════════════════════════════════
# XÂY DỰNG MODEL (v3: 2 tầng BiLSTM nhỏ + attention + 1 dense)
# ══════════════════════════════════════════════
def build_model(num_classes, seq_len=SEQUENCE_LEN, n_feat=N_FEATURES):
    inputs = tf.keras.Input(shape=(seq_len, n_feat), name='keypoints')

    # ── Khối 1: BiLSTM (64 units mỗi chiều = 128 tổng) ──
    x = Bidirectional(LSTM(
        LSTM_UNITS_1,
        return_sequences=True,
        dropout=LSTM_DROPOUT,
        recurrent_dropout=RECURRENT_DROP,
        kernel_regularizer=l2(L2_REG)
    ), name='bilstm_1')(inputs)
    x = BatchNormalization()(x)

    # ── Khối 2: BiLSTM (64 units mỗi chiều) — tầng cuối trước attention ──
    x = Bidirectional(LSTM(
        LSTM_UNITS_2,
        return_sequences=True,
        dropout=LSTM_DROPOUT,
        recurrent_dropout=RECURRENT_DROP,
        kernel_regularizer=l2(L2_REG)
    ), name='bilstm_2')(x)
    x = BatchNormalization()(x)

    # ── Temporal Attention (giữ lại: rẻ về tham số, giúp mô hình tự học
    #    frame nào trong chuỗi quan trọng nhất cho từng từ) ──
    x = TemporalAttention(name='temporal_attention')(x)
    x = BatchNormalization()(x)

    # ── Classifier head (chỉ còn 1 tầng Dense, đủ cho 20-30 lớp) ──
    x = Dense(DENSE_1_UNITS, activation='relu',
              kernel_regularizer=l2(L2_REG))(x)
    x = Dropout(DENSE_DROPOUT)(x)
    x = BatchNormalization()(x)

    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)

    return tf.keras.Model(inputs, outputs, name='VSL_BiLSTM_Attention_v3_small')


model = build_model(NUM_CLASSES)
model.summary()

# Kiểm tra số tham số
total_params = model.count_params()
print(f'\nTổng tham số: {total_params:,}')


In [ ]:
# ══════════════════════════════════════════════
# COSINE DECAY WITH LINEAR WARMUP
# ══════════════════════════════════════════════
total_steps  = EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch

class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, min_lr, warmup_steps, total_steps):
        super().__init__()
        self.base_lr      = base_lr
        self.min_lr       = min_lr
        self.warmup_steps = float(warmup_steps)
        self.total_steps  = float(total_steps)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_lr = self.base_lr * (step / self.warmup_steps)
        progress  = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        progress  = tf.clip_by_value(progress, 0.0, 1.0)
        cosine_lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (
            1.0 + tf.math.cos(math.pi * progress)
        )
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            'base_lr': self.base_lr, 'min_lr': self.min_lr,
            'warmup_steps': self.warmup_steps, 'total_steps': self.total_steps
        }

lr_schedule = WarmupCosineDecay(BASE_LR, MIN_LR, warmup_steps, total_steps)

# ── Label smoothing THỰC SỰ áp dụng (v2) ──────────────────────────────
# v1 dùng SparseCategoricalCrossentropy (không hỗ trợ label_smoothing)
# v2 chuyển sang CategoricalCrossentropy + one-hot labels trong pipeline
loss_fn = tf.keras.losses.CategoricalCrossentropy(
    label_smoothing=LABEL_SMOOTHING,   
    from_logits=False
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=1.0),
    loss=loss_fn,
    metrics=['accuracy']
)
print('Model compiled.')
print(f'Label smoothing: {LABEL_SMOOTHING}  |  L2 reg: {L2_REG}')
print(f'LSTM units: {LSTM_UNITS_1}/{LSTM_UNITS_2} × 2 (BiDir)  |  Dense: {DENSE_1_UNITS}')
print(f'LSTM dropout: {LSTM_DROPOUT}  |  Recurrent dropout: {RECURRENT_DROP}')
print(f'Dense dropout: {DENSE_DROPOUT}')


In [8]:
# ══════════════════════════════════════════════
# CALLBACKS
# ══════════════════════════════════════════════
import shutil
from datetime import datetime

checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_model.keras')
run_id          = datetime.now().strftime('%Y%m%d_%H%M%S')
run_log_dir     = os.path.join(LOG_DIR, run_id)
os.makedirs(run_log_dir, exist_ok=True)
print(f'TensorBoard log dir: {run_log_dir}')

callbacks = [
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',       # [FIX] val_acc đã 100% → dùng val_loss
        save_best_only=True,
        save_weights_only=False,
        mode='min',               # [FIX] loss càng nhỏ càng tốt
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',       # [FIX] cùng monitor với checkpoint
        patience=10,
        restore_best_weights=True,
        mode='min',               # [FIX]
        verbose=1
    ),
    # [FIX] ReduceLROnPlateau đã bị XÓA — xung đột với WarmupCosineDecay.
    # LR đã được quản lý hoàn toàn bởi WarmupCosineDecay trong lr_schedule.
    TensorBoard(
        log_dir=run_log_dir,
        histogram_freq=0,
        update_freq='epoch'
    ),
]
print('Callbacks ready.')
print('[INFO] ReduceLROnPlateau đã bị tắt — LR do WarmupCosineDecay điều khiển.')


TensorBoard log dir: Models/logs\20260530_082836
Callbacks ready.


In [9]:
# ══════════════════════════════════════════════
# RESUME FROM CHECKPOINT (nếu có)
# ══════════════════════════════════════════════
resume_files = glob.glob(os.path.join(CHECKPOINT_DIR, '*.keras'))
if resume_files:
    latest = max(resume_files, key=os.path.getctime)
    print(f'▶ Resume từ checkpoint: {latest}')
    model = tf.keras.models.load_model(
        latest,
        custom_objects={'TemporalAttention': TemporalAttention,
                        'WarmupCosineDecay': WarmupCosineDecay}
    )
else:
    print('▶ Bắt đầu training từ đầu.')


▶ Bắt đầu training từ đầu.


In [ ]:
# ══════════════════════════════════════════════
# TRAINING
# ══════════════════════════════════════════════
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_ds,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)
print('\nTraining hoàn tất!')


In [ ]:
# ══════════════════════════════════════════════
# EVALUATE TRÊN TEST SET
# ══════════════════════════════════════════════
print('\n── Đánh giá trên test set ──')
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f'Test Loss    : {test_loss:.4f}')
print(f'Test Accuracy: {test_acc*100:.2f}%')

# Kiểm tra gap train vs val (chỉ số overfit)
train_acc_final = history.history['accuracy'][-1]
val_acc_final   = history.history['val_accuracy'][-1]
gap = train_acc_final - val_acc_final
print(f'\nTrain Acc (epoch cuối): {train_acc_final*100:.2f}%')
print(f'Val   Acc (epoch cuối): {val_acc_final*100:.2f}%')

model.save(MODEL_SAVE_PATH)
print(f'\n✅ Model đã lưu tại: {MODEL_SAVE_PATH}')



── Đánh giá trên test set ──
4/4 [==============================] - 1s 187ms/step - loss: 0.7266 - accuracy: 1.0000
Test Loss    : 0.7266
Test Accuracy: 100.00%

Train Acc (epoch cuối): 99.37%
Val   Acc (epoch cuối): 100.00%

✅ Model đã lưu tại: final_model1.keras


In [ ]:
# ══════════════════════════════════════════════
# PLOT TRAINING HISTORY
# ══════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Acc')
axes[0].plot(history.history['val_accuracy'], label='Val Acc')
axes[0].set_title('Accuracy — Train vs Val')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss — Train vs Val')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('VSL BiLSTM-Attention v2 (Anti-Overfit)', fontsize=13)
plt.tight_layout()
os.makedirs(PLOT_DIR, exist_ok=True)
plt.savefig(os.path.join(PLOT_DIR, 'training_history_v2.png'), dpi=150)
plt.show()
print('Plot saved to Models/training_history_v2.png')


In [ ]:
# ══════════════════════════════════════════════
# PER-CLASS ACCURACY + CONFUSION MATRIX
# ══════════════════════════════════════════════
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# ── 1. Lấy nhãn thực và nhãn dự đoán từ test set ──
y_true, y_pred = [], []

for seqs, labels in test_ds:
    preds = model.predict(seqs, verbose=0)
    y_pred.extend(np.argmax(preds,  axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ── 2. Tên nhãn từ label_map ──
idx2label  = {v: k for k, v in label_map.items()}
class_names = [idx2label[i] for i in range(NUM_CLASSES)]

# ── 3. Classification report (precision / recall / f1 từng lớp) ──
print("\n── Per-class Report ──")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ── 4. Confusion matrix ──
cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cm_pct,
    annot=True, fmt='.1f', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5, linecolor='gray',
    ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True',      fontsize=12)
ax.set_title('Confusion Matrix (% theo hàng)', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()

os.makedirs(PLOT_DIR, exist_ok=True)
plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print(f'✅ Confusion matrix đã lưu tại: {PLOT_DIR}/confusion_matrix.png')
